# 2. ES 15min Mean Reversion - Parameter Optimization
## Systematic Trading Approach following TIS Framework

This notebook performs a systematic parameter sweep to find the optimal configuration for the ES 15min Mean Reversion hypothesis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from datetime import time
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Load data (same as before)
mongodb_uri = os.environ.get('MONGODB_URI_FUTURES_3')
if not mongodb_uri:
    raise ValueError("MONGODB_URI_FUTURES_3 environment variable not set")

client = MongoClient(mongodb_uri)
db_name = os.getenv("MONGODB_DB", "market_data")
db = client[db_name]
collection = db['es_15min']

projection = {'_id': 0, 'timestamp': 1, 'open': 1, 'high': 1, 'low': 1, 'close': 1, 'volume': 1}
cursor = collection.find({}, projection)
df_raw = pd.DataFrame(list(cursor))

df_raw['time'] = pd.to_datetime(df_raw['timestamp'])
df_raw.set_index('time', inplace=True)
df_raw.drop(columns=['timestamp'], inplace=True)
df_raw.sort_index(inplace=True)

df_15min = df_raw[['open', 'high', 'low', 'close', 'volume']].copy()

print(f"Data loaded: {len(df_15min)} bars")

In [ ]:
def compute_rsi(data, period=2):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def compute_atr(df, period=14):
    high = df['high']
    low = df['low']
    close = df['close']
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    return atr

def run_backtest(df, entry_col_long, entry_col_short, stop_pct=None, target_pct=None, atr_multiplier_stop=None, atr_multiplier_target=None, time_stop_hour=15, time_stop_minute=50):
    trades = []
    position = None
    entry_price = None
    entry_time = None
    entry_type = None
    atr_value = None
    
    for idx, row in df.iterrows():
        if position is not None:
            # Time stop
            if idx.hour > time_stop_hour or (idx.hour == time_stop_hour and idx.minute >= time_stop_minute):
                exit_price = row['close']
                exit_time = idx
                reason = 'time_stop'
                trade = {
                    'entry_time': entry_time,
                    'exit_time': exit_time,
                    'entry_price': entry_price,
                    'exit_price': exit_price,
                    'type': entry_type,
                    'reason': reason
                }
                trades.append(trade)
                position = None
                continue
            
            # ATR-based stop/target (if using ATR)
            if atr_multiplier_stop is not None and atr_value is not None:
                stop_dist = atr_value * atr_multiplier_stop
                target_dist = atr_value * atr_multiplier_target if atr_multiplier_target is not None else stop_dist * 1.5
            else:
                stop_dist = entry_price * (stop_pct/100) if stop_pct is not None else None
                target_dist = entry_price * (target_pct/100) if target_pct is not None else None
            
            if stop_dist is None or target_dist is None:
                break
            
            if entry_type == 'long':
                if row['low'] <= entry_price - stop_dist:
                    exit_price = entry_price - stop_dist
                    reason = 'stop_loss'
                elif row['high'] >= entry_price + target_dist:
                    exit_price = entry_price + target_dist
                    reason = 'take_profit'
                else:
                    continue
            else:
                if row['high'] >= entry_price + stop_dist:
                    exit_price = entry_price + stop_dist
                    reason = 'stop_loss'
                elif row['low'] <= entry_price - target_dist:
                    exit_price = entry_price - target_dist
                    reason = 'take_profit'
                else:
                    continue
            
            exit_time = idx
            trade = {
                'entry_time': entry_time,
                'exit_time': exit_time,
                'entry_price': entry_price,
                'exit_price': exit_price,
                'type': entry_type,
                'reason': reason
            }
            trades.append(trade)
            position = None
            continue
        
        if row[entry_col_long]:
            position = 'long'
            entry_price = row['close']
            entry_time = idx
            entry_type = 'long'
            atr_value = row['atr'] if 'atr' in row else None
        elif row[entry_col_short]:
            position = 'short'
            entry_price = row['close']
            entry_time = idx
            entry_type = 'short'
            atr_value = row['atr'] if 'atr' in row else None
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        return trades_df
    
    trades_df['pnl'] = np.where(trades_df['type'] == 'long',
                               (trades_df['exit_price'] - trades_df['entry_price']) / trades_df['entry_price'] * 100,
                               (trades_df['entry_price'] - trades_df['exit_price']) / trades_df['entry_price'] * 100)
    
    trades_df['win'] = trades_df['pnl'] > 0
    return trades_df

In [ ]:
# Prepare data with indicators
df = df_15min.copy()
df['close_pct_change'] = df['close'].pct_change() * 100
df['volume_ma20'] = df['volume'].rolling(20).mean()
df['volume_ratio'] = df['volume'] / df['volume_ma20']
df['rsi2'] = compute_rsi(df['close'], period=2)
df['atr'] = compute_atr(df, period=14)

df['hour'] = df.index.hour
df['minute'] = df.index.minute
df['dayofweek'] = df.index.dayofweek

In [ ]:
# Parameter sweep
entry_thresholds = [0.5, 0.7, 1.0]
rsi_thresholds = [15, 20, 25]
atr_stop_multipliers = [1.5, 2.0, 2.5]
atr_target_multipliers = [2.5, 3.0, 3.5]
time_stop_hours = [15, 16]
time_stop_minutes = [50, 0]

results = []
for entry_threshold, rsi_threshold, atr_stop, atr_target, time_hour, time_min in product(
    entry_thresholds, rsi_thresholds, atr_stop_multipliers, atr_target_multipliers, time_stop_hours, time_stop_minutes
):
    # Entry conditions
    df['long_entry'] = (df['close_pct_change'] < -entry_threshold) & (df['rsi2'] < rsi_threshold) & (df['volume_ratio'] > 1.2) & (df['hour'] >= 9) & (df['hour'] <= 15) & (df['dayofweek'].isin([1,2,3,4]))
    df['short_entry'] = (df['close_pct_change'] > entry_threshold) & (df['rsi2'] > (100 - rsi_threshold)) & (df['volume_ratio'] > 1.2) & (df['hour'] >= 9) & (df['hour'] <= 15) & (df['dayofweek'].isin([1,2,3,4]))
    
    # Run backtest
    trades = run_backtest(df, 'long_entry', 'short_entry', 
                         atr_multiplier_stop=atr_stop, 
                         atr_multiplier_target=atr_target,
                         time_stop_hour=time_hour, 
                         time_stop_minute=time_min)
    
    if len(trades) == 0:
        continue
    
    win_rate = trades['win'].mean() * 100
    total_pnl = trades['pnl'].sum()
    profit_factor = abs(trades[trades['win']]['pnl'].sum() / trades[~trades['win']]['pnl'].sum()) if len(trades[~trades['win']]) > 0 else np.inf
    expectancy = trades['pnl'].mean()
    num_trades = len(trades)
    
    results.append({
        'entry_threshold': entry_threshold,
        'rsi_threshold': rsi_threshold,
        'atr_stop': atr_stop,
        'atr_target': atr_target,
        'time_stop_hour': time_hour,
        'time_stop_minute': time_min,
        'num_trades': num_trades,
        'win_rate': win_rate,
        'total_pnl': total_pnl,
        'profit_factor': profit_factor,
        'expectancy': expectancy
    })

results_df = pd.DataFrame(results)

In [ ]:
# Display top 10 results by profit factor
top_results = results_df.sort_values('profit_factor', ascending=False).head(10)
print("Top 10 configurations by Profit Factor:")
print(top_results[['entry_threshold', 'rsi_threshold', 'atr_stop', 'atr_target', 'time_stop_hour', 'time_stop_minute', 'num_trades', 'win_rate', 'profit_factor', 'expectancy']].to_string(index=False))

In [ ]:
# Display top 10 by expectancy
top_expectancy = results_df.sort_values('expectancy', ascending=False).head(10)
print("\nTop 10 configurations by Expectancy:")
print(top_expectancy[['entry_threshold', 'rsi_threshold', 'atr_stop', 'atr_target', 'time_stop_hour', 'time_stop_minute', 'num_trades', 'win_rate', 'profit_factor', 'expectancy']].to_string(index=False))